<a href="https://colab.research.google.com/github/fhmahi957/Licence-plate-NightVision/blob/main/Licence_plate_NightVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
from google.colab import drive
import os
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a folder for your models
save_dir = '/content/drive/MyDrive/CSE414_Project2/models'
os.makedirs(save_dir, exist_ok=True)

# 3. Save the trained model weights to Drive
model_path = os.path.join(save_dir, 'faster_rcnn_baseline.pth')
torch.save(model.state_dict(), model_path)

print(f"SUCCESS! Model saved to Google Drive at: {model_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SUCCESS! Model saved to Google Drive at: /content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline.pth


In [4]:
# Cell 2
# Load cell (run this first)

# 1. Mount Drive and Import Libraries
from google.colab import drive
import torch
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

drive.mount('/content/drive')

# 2. Recreate the Model Architecture (The "Empty Kitchen")
# We must define the exact same architecture we trained
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
# NUM_CLASSES was 4 (Background + 3 objects from your dataset)
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 4)

# 3. Load the Weights from Drive (Bringing the "Cake" out of the fridge)
model_path = '/content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline.pth'
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

# 4. Set to Evaluation Mode
model.eval()
print("SUCCESS! Model loaded from Google Drive and ready for testing.")

Mounted at /content/drive
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 81.7MB/s]


SUCCESS! Model loaded from Google Drive and ready for testing.


In [5]:
# Cell 3
# Evaluation code (run this 2nd)

import os
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from torchmetrics.detection import MeanAveragePrecision
from tqdm import tqdm

# Paths (to fix data leakage)
NIGHT_IMG_DIR = '/content/night_test_images'
LBL_DIR = '/content/test_labels' # Using test labels, not source labels

# 1. Define the Nighttime Dataset
# We iterate through the ORIGINAL filenames, but load the NIGHTTIME images
class NighttimeDataset(Dataset):
    def __init__(self, night_img_dir, lbl_dir):
        self.night_img_dir = night_img_dir
        self.lbl_dir = lbl_dir
        # Get list of original filenames (e.g., 'car1.jpg')
        # We assume the night images are named 'night_car1.jpg'
        self.original_images = sorted([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])

    def __len__(self):
        return len(self.original_images)

    def __getitem__(self, idx):
        orig_name = self.original_images[idx] # e.g., 'car1.txt'
        img_name = orig_name.replace('.txt', '.jpg') # e.g., 'car1.jpg'
        night_img_name = f"night_{img_name}" # e.g., 'night_car1.jpg'

        img_path = os.path.join(self.night_img_dir, night_img_name)
        lbl_path = os.path.join(self.lbl_dir, orig_name)

        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        boxes = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center = float(parts[1])
                        y_center = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])

                        x1 = (x_center - w / 2) * width
                        y1 = (y_center - h / 2) * height
                        x2 = (x_center + w / 2) * width
                        y2 = (y_center + h / 2) * height

                        # Clamp
                        x1 = max(0, x1); y1 = max(0, y1)
                        x2 = min(width, x2); y2 = min(height, y2)

                        if x2 > x1 and y2 > y1:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(class_id + 1) # 1-indexed

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
        }

        # Transform to tensor (same as training)
        transform = T.Compose([T.ToTensor()])
        image = transform(image)

        return image, target

# 2. Setup Data Loader
dataset = NighttimeDataset(NIGHT_IMG_DIR, LBL_DIR)
# Batch size 1 is best for evaluation metrics
data_loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval() # Crucial for evaluation

# 3. Initialize Metric
metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy")

print("Starting Domain Shift Analysis (Testing Clean Model on Nighttime Data)...")
print("This measures the 'Before' performance.\n")

# 4. Evaluation Loop
with torch.no_grad():
    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Get predictions
        predictions = model(images)

        # Format for torchmetrics
        # predictions is a list of dicts: [{'boxes':..., 'labels':..., 'scores':...}]
        preds_for_metric = [{
            'boxes': p['boxes'].cpu(),
            'labels': p['labels'].cpu(),
            'scores': p['scores'].cpu()
        } for p in predictions]

        targets_for_metric = [{
            'boxes': t['boxes'].cpu(),
            'labels': t['labels'].cpu()
        } for t in targets]

        # Update metric
        metric.update(preds_for_metric, targets_for_metric)

# 5. Compute and Print Results
result = metric.compute()
print("\n DOMAIN SHIFT ANALYSIS RESULTS")
print(f"mAP (mean Average Precision): {result['map'].item():.4f}")
print(f"mAP @ 0.5 IoU: {result['map_50'].item():.4f}")
print(f"mAP @ 0.75 IoU: {result['map_75'].item():.4f}")

print("Note: A low mAP here proves that Domain Shift is a problem!")


Starting Domain Shift Analysis (Testing Clean Model on Nighttime Data)...
This measures the 'Before' performance.



Evaluating: 100%|██████████| 99/99 [00:05<00:00, 18.47it/s]



 DOMAIN SHIFT ANALYSIS RESULTS
mAP (mean Average Precision): 0.2154
mAP @ 0.5 IoU: 0.3234
mAP @ 0.75 IoU: 0.2376
Note: A low mAP here proves that Domain Shift is a problem!


In [1]:
# Cell mr bean

# Installing necessary libraries
!pip install ultralytics albumentations opencv-python-headless matplotlib seaborn tqdm pyyaml torchmetrics -q

In [ ]:
# Cell 4
import os
import cv2
import torch
import albumentations as A
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Check if the GPU is connected and ready
print("Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"Is GPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not available. Please go to Runtime > Change runtime type and select T4 GPU.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment setup complete!
PyTorch version: 2.11.0+cu130
Is GPU available: True
GPU Name: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os


print("Contents of MyDrive:")
print(os.listdir('/content/drive/MyDrive/'))


project_path = '/content/drive/MyDrive/CSE414_Project2'
if os.path.exists(project_path):
    print(f"\nFound project folder! Contents:")
    print(os.listdir(project_path))
else:
    print("\nCSE414_Project2 folder not found in MyDrive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Contents of MyDrive:
['wallpaperflare.com_wallpaper.jpg', 'Yearbook.gslides', 'Untitled presentation (1).gslides', 'Untitled presentation.gslides', 'Google AI Studio', 'Untitled document (2).gdoc', '23101182_report2 .gdoc', 'Untitled document (1).gdoc', '5.gdoc', '7.gdoc', 'Queries for B2B automation platform.gdoc', '23101182,23101183,23101186,23101189.docx', 'Overview of Smart Water Tank project .gdoc', ' 23101186_Crypto-Algo-Imply.gdoc', '23101182_Crypto-Algo-Imply.gdoc', 'Lead Capture (updated code files).gdoc', 'Dashboard & State api (updated code).gdoc', 'Auth and rbac(updated code).gdoc', 'API-Documentation (Group-3).gdoc', 'Apology mail for delay .gsheet', 'Scheduled_Emails.gsheet', 'Untitled document.gdoc', 'Classroom', 'Tasnim Rahman Tayeba.gslides', 'Fire Fighting Robot (Group-5).gslides', 'Colab Notebooks', 'CSE414_Project2']

Found project folder!

In [2]:
#Cell 5

from google.colab import files
import zipfile
import os
import shutil

# 1. Upload zip file from pc
print("Please upload the Roboflow ZIP file from your computer...")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]
print(f"Uploaded: {zip_filename}")

# 2. Creating seperate folder for both test and train
os.makedirs('/content/train_images', exist_ok=True)
os.makedirs('/content/train_labels', exist_ok=True)
os.makedirs('/content/test_images', exist_ok=True)
os.makedirs('/content/test_labels', exist_ok=True)

# 3.Unzip and organize
print("\nUnzipping and splitting data...")
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset_raw')

base_path = '/content/dataset_raw'

for root, dirs, files in os.walk(base_path):
    # Train Split
    if 'images' in dirs and 'train' in root:
        for img in os.listdir(os.path.join(root, 'images')):
            if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(os.path.join(root, 'images', img), '/content/train_images/')
    if 'labels' in dirs and 'train' in root:
        for lbl in os.listdir(os.path.join(root, 'labels')):
            if lbl.lower().endswith('.txt'):
                shutil.move(os.path.join(root, 'labels', lbl), '/content/train_labels/')

    # Test Split
    if 'images' in dirs and 'test' in root:
        for img in os.listdir(os.path.join(root, 'images')):
            if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(os.path.join(root, 'images', img), '/content/test_images/')
    if 'labels' in dirs and 'test' in root:
        for lbl in os.listdir(os.path.join(root, 'labels')):
            if lbl.lower().endswith('.txt'):
                shutil.move(os.path.join(root, 'labels', lbl), '/content/test_labels/')

print(f"\nData Split Done!")
print(f"Train Images: {len(os.listdir('/content/train_images'))}")
print(f"Test Images: {len(os.listdir('/content/test_images'))}")

Please upload the Roboflow ZIP file from your computer...


Saving bangla-license-plate-detection.v1i.yolov8.zip to bangla-license-plate-detection.v1i.yolov8.zip
Uploaded: bangla-license-plate-detection.v1i.yolov8.zip

Unzipping and splitting data...

Data Split Done!
Train Images: 746
Test Images: 99


In [3]:
#Cell 6

# taking new images form test folder
# creating distorted image from clean image
import os
import cv2
import albumentations as A
from tqdm import tqdm

# change : creating night image from test image
CLEAN_TEST_DIR = '/content/test_images'
NIGHT_TEST_DIR = '/content/night_test_images'
os.makedirs(NIGHT_TEST_DIR, exist_ok=True)

nighttime_transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.6, -0.2), contrast_limit=(-0.4, 0.0), p=0.9),
    A.GaussianBlur(blur_limit=(3, 7), p=0.6),
    A.MotionBlur(blur_limit=(5, 11), p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.5)
])

print("Generating synthetic nighttime images from TEST set only...")
image_files = [f for f in os.listdir(CLEAN_TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for filename in tqdm(image_files, desc="Processing test images"):
    img_path = os.path.join(CLEAN_TEST_DIR, filename)
    image = cv2.imread(img_path)
    if image is None: continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    transformed = nighttime_transform(image=image_rgb)
    degraded_image_bgr = cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR)

    save_path = os.path.join(NIGHT_TEST_DIR, f"night_{filename}")
    cv2.imwrite(save_path, degraded_image_bgr)

print(f"\n Success! Generated {len(image_files)} nighttime images in /content/night_test_images")

Generating synthetic nighttime images from TEST set only...


Processing test images: 100%|██████████| 99/99 [00:00<00:00, 111.39it/s]


 Success! Generated 99 nighttime images in /content/night_test_images


In [ ]:
#Cell 7

#checking class ID's
import os

LBL_DIR = '/content/source_labels'
class_ids = set()

for lbl_file in os.listdir(LBL_DIR):
    if lbl_file.endswith('.txt'):
        with open(os.path.join(LBL_DIR, lbl_file), 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 1:
                    class_ids.add(int(parts[0]))

print(f"Found class IDs in dataset: {sorted(class_ids)}")
print(f"Max class ID: {max(class_ids) if class_ids else 0}")

Found class IDs in dataset: [0, 1, 2]
Max class ID: 2


In [ ]:
# Cell 8
import os
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from PIL import Image

# Change : Only using train data
IMG_DIR = '/content/train_images'
LBL_DIR = '/content/train_labels'

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.lbl_dir, img_name.rsplit('.', 1)[0] + '.txt')

        image = Image.open(img_path).convert("RGB")
        width, height = image.size
        boxes, labels = [], []

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x1 = (float(parts[1]) - float(parts[3])/2) * width
                        y1 = (float(parts[2]) - float(parts[4])/2) * height
                        x2 = (float(parts[1]) + float(parts[3])/2) * width
                        y2 = (float(parts[2]) + float(parts[4])/2) * height

                        x1, y1, x2, y2 = max(0,x1), max(0,y1), min(width,x2), min(height,y2)
                        if x2 > x1 and y2 > y1:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(class_id + 1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([idx]),
                  'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
                  'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)}

        return torchvision.transforms.ToTensor()(image), target

dataset = LicensePlateDataset(IMG_DIR, LBL_DIR)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 4) # 4 classes
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
# Change : Reduced learning rate and increase Epoch
optimizer = optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

num_epochs = 20
print(f"Starting training for {num_epochs} epochs on Train Set...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()
    lr_scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss/len(data_loader):.4f}")

print("\nTraining complete!")

Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth


100%|██████████| 74.2M/74.2M [00:00<00:00, 191MB/s]


Starting training for 20 epochs on Train Set...
Epoch 1/20 | Loss: 0.3559
Epoch 2/20 | Loss: 0.4293
Epoch 3/20 | Loss: 0.4855
Epoch 4/20 | Loss: 0.4829
Epoch 5/20 | Loss: 0.4835
Epoch 6/20 | Loss: 0.4676
Epoch 7/20 | Loss: 0.4659
Epoch 8/20 | Loss: 0.4614
Epoch 9/20 | Loss: 0.4876
Epoch 10/20 | Loss: 0.4513
Epoch 11/20 | Loss: 0.3837
Epoch 12/20 | Loss: 0.3784
Epoch 13/20 | Loss: 0.3621
Epoch 14/20 | Loss: 0.3491
Epoch 15/20 | Loss: 0.3543
Epoch 16/20 | Loss: 0.3614
Epoch 17/20 | Loss: 0.3527
Epoch 18/20 | Loss: 0.3450
Epoch 19/20 | Loss: 0.3415
Epoch 20/20 | Loss: 0.3413

Training complete!
